# Pilot experiment on Colab (T4)
3 models × 4 datasets × 15 samples. Three parts: **A. setup**, **B. run the experiment**, **C. look at the results**.

Before you start: `pilot_data.zip` must be in Google Drive at `MyDrive/TEZ/pilot_data.zip`
(made locally with `python src/pack_pilot_data.py`). Runtime → Change runtime type → **T4 GPU**.
Raw outputs are written to `MyDrive/TEZ/results/raw/`, line by line, so nothing is lost if Colab disconnects.

## A. Setup (about 5 minutes)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "https://github.com/nilucmak/Structured-Report-Generation-from-Aerial-and-Satellite-Images-with-VLMs.git"
DRIVE_DIR = "/content/drive/MyDrive/TEZ"        # contains pilot_data.zip; results are written here

In [ ]:
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
# Always use the latest code from GitHub (this also undoes the Colab-only config edit below)
!git fetch -q origin && git reset -q --hard origin/main
!git log -1 --oneline
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists("/content/pilot_data"):
    !unzip -q {DRIVE_DIR}/pilot_data.zip -d /content/pilot_data

# configs/config.yaml stays the single settings file; only the two paths are changed in this Colab copy
import yaml
cfg = yaml.safe_load(open("configs/config.yaml"))
cfg["paths"]["data_root"] = "/content/pilot_data"
cfg["paths"]["raw_dir"] = f"{DRIVE_DIR}/results/raw"
yaml.safe_dump(cfg, open("configs/config.yaml", "w"), sort_keys=False)

!python tests/parser_test.py | tail -1
!python src/run_inference.py --model qwen2vl_7b --family native --dry-run | tail -1

## B. Run the experiment (about 2–3 hours)
Runs every model with both prompt families (`native` = the model's own output format, `json` = our report).
The first run of each model also downloads it (Qwen2-VL ~16 GB, Qwen2.5-VL ~7.5 GB, GeoVLM-R1 3 × ~7.5 GB).
**If Colab disconnects:** run part A again, then this cell again. Finished samples are skipped.

In [ ]:
for model in ["qwen2vl_7b", "qwen25vl_3b", "geovlm_r1"]:
    for family in ["native", "json"]:
        !python src/run_inference.py --model {model} --family {family}

## C. Results
These cells can be run at any time, also while part B is still running (they show what is finished).
This is a quick look (a model box counts as matched if it overlaps a ground-truth box with IoU ≥ 0.5,
class ignored). The final metrics come in stage 5.

In [ ]:
import sys
sys.path.insert(0, "src")
import pandas as pd
from config import load_config
import show_results
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 20)
cfg = load_config()
show_results.summary(cfg)

### DIOR (satellite, objects) — green = ground truth, red = model

In [ ]:
show_results.show(cfg, "dior", n=3)

### VisDrone (drone, objects)

In [ ]:
show_results.show(cfg, "visdrone", n=3)

### LEVIR-CD+ (change between two times; boxes on the later image)

In [ ]:
show_results.show(cfg, "levir_cdplus", n=3)

### RSICD (scene description; reference caption vs model captions)

In [ ]:
show_results.show(cfg, "rsicd", n=3)

### Example of a final "what / where / when" report
Built from a model's JSON answer by the report schema: `when` comes only from metadata ("unknown" here),
and every correction is listed in `warnings`.

In [ ]:
import json
for m in ["qwen2vl_7b", "qwen25vl_3b", "geovlm_r1"]:
    print(f"===== {m}")
    print(json.dumps(show_results.example_report(cfg, m, "dior"), indent=1, ensure_ascii=False))

To see more samples, change `n=3` (up to 15).